In [9]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
import json

openai_client = OpenAI()

In [2]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Usually, yes — if the course is still open for enrollment and you meet any prerequisites.\n\nA few things to check:\n- **Enrollment status:** Is registration still open?\n- **Prerequisites:** Do you need prior experience, approval, or placement?\n- **Capacity:** Is the class full or waitlisted?\n- **Deadlines:** Some courses allow late registration only for a short time.\n\nIf you want, I can help you draft a quick message to the instructor or registrar asking to join.'

In [11]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output 

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join eligibility enrollment late join"}', call_id='call_EcSHfJpOxeVakhaBxGaALx0L', name='search', type='function_call', id='fc_038bc7442da882b8006a314acfda50819d8a910124c6d7009f', namespace=None, status='completed')]

In [ ]:


call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [8]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [9]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.  \n\nIf you want a certificate, make sure you submit your project while submissions are still open.'

In [10]:
response.output

[ResponseOutputMessage(id='msg_038bc7442da882b8006a314ad2c298819d9c45e04797583f13', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.  \n\nIf you want a certificate, make sure you submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [3]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [4]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [12]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

print(response.output)
for item in response.output:
    print(item.type)
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join late enrollment registration FAQ"}', call_id='call_7XRFUJGRRuhwOeZiaOEoAr3N', name='search', type='function_call', id='fc_0bc4f58da8c4dfe0006a314f120370819dbc73e5a29ab9d091', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course enrollment late join student FAQ discovered course"}', call_id='call_HFo3OoxeFmsXiEhScC2TscNi', name='search', type='function_call', id='fc_0bc4f58da8c4dfe0006a314f120380819d8156dc65b66d5ba5', namespace=None, status='completed')]
function_call
function_call: search {"query":"join course discovered course can I join late enrollment registration FAQ"}
function_call
function_call: search {"query":"course enrollment late join student FAQ discovered course"}


In [14]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration FAQ"}', call_id='call_wFFQf6yM6HXEXdDrLP8zSYDT', name='search', type='function_call', id='fc_08e84dfe49af45aa006a314cab198481a3bb0dd1402bce5f16', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment join after course started late join FAQ"}', call_id=

In [14]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join the course.

If your goal is a certificate, you’ll need to submit your project while submissions are still open. If you’re just learning, you can start anytime with the course materials.

If you want, I can also help you figure out the best way to start or explain the certificate requirements.


In [17]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]


it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)
    print(response.output)
    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment registration late join FAQ"}', call_id='call_p5RR3BNWpDZU9Xq429aSyWGu', name='search', type='function_call', id='fc_00edaed613f65b73006a31526db384819eaebe016a9ff60cba', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course discovered can I join it late enrollment FAQ"}', call_id='call_qR6M1eqTYJR7GJKb0WTItJF4', name='search', type='function_call', id='fc_00edaed613f65b73006a31526db39c819e90eeab1fafeaf98f', namespace=None, status='completed')]
function_call: search {"query":"join course discovered course can I join enrollment registration late join FAQ"}
function_call: search {"query":"course discovered can I join it late enrollment FAQ"}
iteration #2...
[ResponseOutputMessage(id='msg_00edaed613f65b73006a31526e92c8819e84dbafa1604577e5', content=[ResponseOutputText(annotations=[], text="Yes — you can still join the course.\n\nIf you want a